# 12 · Adaptation after a distribution switch

**What this shows.** What happens to the criterion (PSE) when the stimulus distribution changes between
blocks — Uniform → Hard-B → Hard-A → Hard-B → Hard-A — in the pre-opto cohorts: the size of the shift, how fast it
converges within a block, what carries over overnight, and the per-session psychometric parameters. This is the
behaviour the PPC manipulation is meant to perturb. Notebook 13 sets these blocks beside the daily alternation of
the opto mice (chapter 3, Fig 3.6).

**What produced it.** A `switch_adaptation` run:

```bash
sc-reports switch-adaptation --cohort behaviour1-cohort
```

which writes, under `switch_adaptation/<cohort>/<run_id>/switches/`: `switches.csv` (one row per switch
and stat), `pre_post.csv` (PSE before and after each switch, with the normative PSE), `convergence.csv`
(the criterion against trials since the switch, per method), `sessions.csv` (per-session stats in block order),
`overnight.csv` (end-of-day to next-morning criterion), `animals.csv` (one number per mouse and kind of switch),
`bias.csv` (the one bias check), `psychometrics.csv` + `psychometric_curves.csv` (per-phase fits), and PDFs per
animal plus `summary_switches.pdf`. A run can hold daily-alternation animals too (`design` = daily); this
notebook reads the blocks.

**Design note.** This cohort switched in blocks of days; the opto cohort alternated Hard-A and Hard-B daily
(notebooks 13 and 21). §6 asks the plain question for the blocks — did the criterion move with the distribution —
and its answer decides whether the post-shift opto prediction can be tested on these designs at all.


In [ ]:
from nb_setup import *
banner()
run = open_run('switch_adaptation', SWITCH_COHORT)
T = load_switches(run)
T = {k: (v[v['design'] == 'blocked'].reset_index(drop=True) if 'design' in v else v) for k, v in T.items()}
biased = sorted(set(T['bias'].loc[T['bias']['biased_animal'].astype(bool), 'animal'])) if 'bias' in T else []
{k: v.shape for k, v in T.items()}


## 1 · One animal: where the switches are, and the pipeline

`behaviour/adaptation.py` finds switches from the session table (`animal_switches`), then
`compute_switches` fits the criterion before and after each one and the convergence curve within the new block.


In [ ]:
from sound_categorisation.behaviour.adaptation import animal_switches

exp = load_experiment()
aid = sorted(T['switches'].animal.unique())[0]
animal = exp.animals[aid]
sessions, switches = animal_switches(animal, stage=None if SYNTHETIC else 'Full_Task_Cont', min_block_trials=300)
print(f'{aid}: {len(sessions)} sessions, {len(switches)} switches')
pd.DataFrame(switches).head(10) if switches else 'no switches'

## 2 · Pre/post PSE per switch, every animal

`pre_post.csv`: the PSE fitted on the last trials before the switch and the first after it, against the
normative PSE for the new distribution (the criterion an ideal observer of that distribution would adopt).

In [ ]:
pp = T['pre_post']
display(pp[['animal', 'switch_idx', 'transition', 'pse_pre', 'pse_post', 'normative_pse', 'delta']].round(3))

fig, ax = plt.subplots(figsize=(6, 3.5))
for tr, sub in pp.groupby('transition'):
    ax.plot([0, 1], [sub['pse_pre'], sub['pse_post']], color='0.6', lw=0.8, alpha=0.6)
    ax.scatter([1] * len(sub), sub['pse_post'], label=tr, s=30, zorder=3)
ax.set(xticks=[0, 1], xticklabels=['pre', 'post'], ylabel='PSE', title='PSE before and after each switch')
ax.axhline(0, color='0.8', lw=0.6); ax.legend(fontsize=8)

## 2b · Who adapts, who doesn't, who goes the wrong way

For each switch, `sign(delta)` is compared with the direction the switch moves the ideal observer's criterion
(from the previous distribution's normative PSE to the new one): **toward** (shifted the right way), **none**
(|delta| below a tolerance), **reverse** (the wrong way), **no fit** (a PSE missing, or beyond ±1, outside the
stimulus range: the fit failed, so the switch says nothing). Animals the bias check drops (`adaptation.BiasRule`:
they stick to one side whatever the distribution) are listed but not tallied.

In [ ]:
TOL = 0.05                                             # |delta| below this counts as no shift
pp = T['pre_post'].sort_values(['animal', 'switch_idx']).copy()
pse = pp[['pse_pre', 'pse_post']]
no_fit = pse.isna().any(axis=1) | (pse.abs() > 1).any(axis=1)
previous_normative = pp.groupby('animal')['normative_pse'].shift(1).fillna(0.0)
expected = np.sign(pp['normative_pse'] - previous_normative)
pp['direction'] = np.where(no_fit, 'no fit', np.where(pp['delta'].abs() < TOL, 'none',
                           np.where(np.sign(pp['delta']) == expected, 'toward', 'reverse')))
pp['biased'] = pp['animal'].isin(biased)

tally = pp[~pp['biased']].pivot_table(index='animal', columns='direction', values='switch_idx', aggfunc='count',
                                      fill_value=0)
for col in ('toward', 'none', 'reverse', 'no fit'):
    if col not in tally:
        tally[col] = 0
tally['n_switches'] = tally[['toward', 'none', 'reverse']].sum(axis=1)
tally['fraction_toward'] = (tally['toward'] / tally['n_switches'].replace(0, np.nan)).round(2)
display(tally[['n_switches', 'toward', 'none', 'reverse', 'no fit', 'fraction_toward']].sort_values('fraction_toward'))
fitted = pp[~pp['biased'] & (pp['direction'] != 'no fit')]
print(f"switches with a fit (animals the bias check keeps): {len(fitted)}; "
      f"toward {(fitted.direction == 'toward').sum()}, none {(fitted.direction == 'none').sum()}, "
      f"reverse {(fitted.direction == 'reverse').sum()}")
print('dropped by the bias check:', biased or 'none')
pp[['animal', 'switch_idx', 'transition', 'pse_pre', 'pse_post', 'normative_pse', 'delta', 'direction', 'biased']].round(3)

## 3 · Convergence within the new block (Fig. 5C)

`convergence.csv`: the criterion as a function of trials since the switch, per switch and method, and
`convergence` = how far the shift has gone toward the normative criterion (0 = still at the criterion before the
switch, 1 = at the normative value). Shown here with the report's measure (`pinned_running`: slope and lapses
fixed at the mouse's expert-Uniform fit, running 50-trial windows, not clipped); the manuscript's clipped recipe
is the `manuscript` method. Mean over the clean animals per transition, individual switches faint.


In [ ]:
cv = T['convergence']
sub = cv[cv['method'] == 'pinned_running']                    # the measure the report uses
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for tr, s in sub[~sub['animal'].isin(biased)].groupby('transition'):
    for (a, k), ss in s.groupby(['animal', 'switch_idx']):
        axes[0].plot(ss['trial'], ss['convergence'], color='0.75', lw=0.6)
    m = s.groupby('trial')['convergence'].mean()
    axes[0].plot(m.index, m.values, lw=2, label=tr)
axes[0].axhline(1, color='0.5', lw=0.6, ls='--'); axes[0].axhline(0, color='0.5', lw=0.6)
axes[0].set(xlabel='trials since switch (window centre)', ylabel='convergence (0 = before, 1 = normative)',
            title='criterion, shape fixed, running 50-trial windows', xscale='log', ylim=(-1.5, 2.5))
axes[0].legend(fontsize=8)
# per-switch trials-to-90% as a distribution
sw = T['switches']
t90 = sw[sw['stat'].str.contains('trials_to_90')]
if len(t90):
    axes[1].hist(t90['value'].dropna(), bins=12, color='0.6')
    axes[1].set(xlabel='trials to 90 % of the shift', ylabel='switches')
plt.tight_layout()


## 4 · Session by session inside a block, and overnight

`sessions.csv` gives every stat per session in block order (`order_in_block`); `overnight.csv` the criterion over
the last 100 trials of a session against the first 100 of the next — does the shift survive the night, or does the
animal drift back toward the previous distribution? `toward_pre` signs the overnight change by the direction of the
switch (> 0 = back towards the previous distribution). It used to be signed by the end-of-session value itself,
which made it positive from noise alone.


In [ ]:
se = T['sessions']
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for (a, k), s in se.groupby(['animal', 'switch_idx']):
    s = s.sort_values('order_in_block')
    axes[0].plot(s['order_in_block'], s['pse_pinned'] - s['pse_pinned'].iloc[0], color='0.6', lw=0.8, alpha=0.6)
axes[0].axhline(0, color='0.5', lw=0.6)
axes[0].set(xlabel='session in block', ylabel='criterion − first session of block', title='within-block drift')
ov = T['overnight']
axes[1].scatter(ov['pse_end'], ov['pse_start_next'], s=20)
lim = np.nanmax(np.abs(ov[['pse_end', 'pse_start_next']].to_numpy())) * 1.1 if len(ov) else 1
axes[1].plot([-lim, lim], [-lim, lim], color='0.6', lw=0.8)
back = int((ov['toward_pre'] > 0).sum())
axes[1].set(xlabel='criterion at end of session', ylabel='criterion at start of next',
            title=f'overnight: {back}/{int(ov["toward_pre"].notna().sum())} pairs moved back toward the previous distribution')
plt.tight_layout()


## 5 · Per-phase psychometrics across animals

The fitted curves per phase (block) for every animal, with the fit parameters. This is the group picture
the opto post-shift experiment compares against.

In [ ]:
ps, pc = T['psychometrics'], T['psychometric_curves']
dists = [d for d in pc['distribution'].unique() if isinstance(d, str)]
fig, axes = plt.subplots(1, len(dists), figsize=(3.6 * len(dists), 3.2), sharey=True, squeeze=False)
for ax, d in zip(axes[0], dists):
    for (a, ph), s in pc[pc['distribution'] == d].groupby(['animal', 'phase']):
        ax.plot(s['x'], s['y'], color='0.6', lw=0.7, alpha=0.6)
    m = pc[pc['distribution'] == d].groupby('x')['y'].mean()
    ax.plot(m.index, m.values, lw=2, color=PALETTE[0])
    ax.axhline(0.5, color='0.8', lw=0.5); ax.axvline(0, color='0.8', lw=0.5)
    ax.set(title=d, xlabel='stimulus')
axes[0, 0].set_ylabel('P(choose B)')
plt.tight_layout()
ps.groupby('distribution')[['mu', 'sigma', 'accuracy', 'hard_accuracy']].agg(['median', 'count']).round(3)

## 6 · In one line: did the criterion follow the distribution?

The median fitted criterion per phase (blocks; animals the bias check keeps), and the difference between the
Hard-A and Hard-B blocks against the difference an ideal observer would show. Adaptation would put the animals'
difference near the ideal one; a shift shared by both Hard phases is not adaptation to either. The blocks are
not counterbalanced (Hard-B first, Hard-A later), so a slow drift of the criterion adds to this difference; the
switch-by-switch tally of §2b is the cleaner test.

In [ ]:
ps = T['psychometrics']
kept = ps[~ps['animal'].isin(biased)] if 'animal' in ps else ps
med = kept.groupby('distribution')['mu'].median()
ideal = pp.loc[pp['normative_pse'] > 0, 'normative_pse'].median() - pp.loc[pp['normative_pse'] < 0, 'normative_pse'].median()
print('median criterion (mu) per phase:', med.round(3).to_dict())
if {'Hard-A', 'Hard-B'} <= set(med.index):
    print(f"Hard-A − Hard-B: animals {med['Hard-A'] - med['Hard-B']:+.3f}, ideal observer {ideal:+.3f}")